# Model 3 — EEG WaveNet 1D Dilated Causal CNN → Transformer + MSM Pretraining
**Multimodal Intraoperative Prediction System — Model 3 of 5**

Encodes brain electrical state from raw EEG waveforms (`BIS/EEG1_WAV`, `BIS/EEG2_WAV`) and BIS scalar channels.
- **Backbone**: WaveNet dilated causal CNN → Transformer encoder
- **Stage 0**: Masked Segment Modelling (MSM) self-supervised pretraining
- **Stage 1**: Supervised fine-tuning for IOH (primary) and N/H (secondary) labels
- **SQI gate**: windows with `BIS/SQI < 70` excluded from loss; EEG embedding zeroed when `SQI < 50`
- **EEG sample rate**: 128 Hz

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. Imports and seed setting
# ─────────────────────────────────────────────────────────────────────────────
from __future__ import annotations
import gc
import logging
import math
import os
import random
import time
from typing import Dict, List, Optional, Tuple

import matplotlib
matplotlib.use("Agg")  # non-interactive backend
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import stft as scipy_stft
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Updated: torch.amp replaces deprecated torch.cuda.amp imports
from torch.amp import GradScaler, autocast

from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import vitaldb  # VitalDB Python package — live download, in-memory only

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. CONFIG dict — ALL hyperparameters, track names, file paths
#    NOTE: EEG sample rate is 128 Hz (not 500 Hz).
#          All derived sample counts updated accordingly.
# ─────────────────────────────────────────────────────────────────────────────
CONFIG: Dict = {
    # ── Identity ─────────────────────────────────────────────────────────────
    "model_name": "model_3",
    "output_file": "model_3.py",
    "log_file": "model_3.log",

    # ── VitalDB tracks ────────────────────────────────────────────────────────
    # Raw EEG at 128 Hz
    "wav_tracks": ["BIS/EEG1_WAV", "BIS/EEG2_WAV"],
    "wav_interval": 1 / 128,        # seconds per sample → 128 Hz

    # BIS scalar channels at 1 Hz
    "scalar_tracks": [
        "BIS/BIS", "BIS/SQI", "BIS/EMG",
        "BIS/SR", "BIS/SEF", "BIS/TOTPOW",
        "Solar8000/ART_MBP",         # needed for IOH label construction
        "Solar8000/HR",              # needed for N/H label
    ],
    "scalar_interval": 1,           # 1 Hz

    # Clinical metadata params loaded via load_clinical_data()
    "clinical_params": [
        "caseid", "anestart", "aneend", "opstart", "opend", "ane_type",
    ],

    # ── Case eligibility ──────────────────────────────────────────────────────
    "min_case_duration_sec": 1800,  # 30 minutes minimum
    "min_sqi_coverage": 0.70,       # >= 70 % of case must have SQI >= 70
    "sqi_train_threshold": 70,      # exclude from loss when SQI < 70
    "sqi_zero_threshold": 50,       # zero-out embedding when SQI < 50

    # ── Window / stride ───────────────────────────────────────────────────────
    "history_sec": 1800,            # 30-min history window → 1800 s at 1 Hz
    "stride_sec": 60,               # 1-minute stride between windows
    "max_nan_frac": 0.20,           # discard window if > 20 % NaN in any channel
    "max_fill_gap_sec": 60,         # forward-fill up to 60 s

    # ── IOH label ─────────────────────────────────────────────────────────────
    "ioh_thresholds_mmhg": 65.0,
    "ioh_min_duration_sec": 60,
    "ioh_horizons_sec": [300, 600, 900],   # t+5, t+10, t+15 min

    # ── N/H label ─────────────────────────────────────────────────────────────
    "nh_n_classes": 3,
    "nh_label_smoothing": 0.15,
    "nh_focal_gamma": 2.0,
    "nh_hr_high": 90,               # bpm threshold for nociception excess
    "nh_bis_low": 60,               # BIS threshold for nociception excess
    "nh_bis_high": 65,              # BIS rising threshold for hypnosis excess
    "nh_hr_rise_frac": 0.15,        # 15 % HR rise for hypnosis excess

    # ── Waveform pre-processing ───────────────────────────────────────────────
    "eeg_fs": 128,                  # Hz — BIS EEG waveform sample rate
    "eeg_window_sec": 30,           # raw EEG per 1-Hz output step: 30 s × 128 = 3840 samples
    "eeg_samples_per_step": 3840,   # 30 s × 128 Hz

    # ── STFT spectral features ────────────────────────────────────────────────
    "stft_win_sec": 4,              # 4-s Hann window
    "stft_overlap": 0.5,            # 50 % overlap
    "stft_bands": {                 # Hz bands → band power features
        "delta": (0.5, 4.0),
        "theta": (4.0, 8.0),
        "alpha": (8.0, 13.0),
        "beta":  (13.0, 30.0),
    },

    # ── BSR ───────────────────────────────────────────────────────────────────
    "bsr_window_sec": 63,           # 63-s BSR window (per spec)
    "bsr_amplitude_uv": 5.0,        # amplitude threshold for suppression

    # ── WaveNet CNN ───────────────────────────────────────────────────────────
    "wavenet_dilations": [
        1, 2, 4, 8, 16, 32, 64, 128,
        1, 2, 4, 8, 16, 32, 64, 128,
    ],                              # 16 layers, two cycles of 8
    "wavenet_kernel_size": 3,
    "wavenet_residual_channels": 64,
    "wavenet_skip_channels": 64,
    "wavenet_cnn_out_dim": 64,      # output feature dim after CNN
    "wavenet_n_output_vectors": 256,  # 3840 samples → pooled to 256 vectors
                                      # (reduced from 512 to match 128 Hz input)

    # ── MSM pretraining ───────────────────────────────────────────────────────
    "msm_mask_prob": 0.20,          # 20 % of raw EEG masked
    "msm_mask_span_sec": 0.5,       # random spans of 0.5 s
    "msm_mask_span_samples": 64,    # 0.5 s × 128 Hz = 64 samples
    "msm_epochs": 20,
    "msm_lr": 1e-4,
    "msm_batch_size": 8,            # small batch — raw EEG is large

    # ── Transformer ───────────────────────────────────────────────────────────
    "transformer_d_model": 128,
    "transformer_n_heads": 4,
    "transformer_n_layers": 4,
    "transformer_ffn_dim": 256,
    "transformer_dropout": 0.1,
    "transformer_n_spectral_features": 9,  # 4 band powers + SEF95 + BSR + 3 BIS scalars (BIS,EMG,SR)

    # ── Supervised training ───────────────────────────────────────────────────
    "sup_epochs": 50,
    "sup_lr": 5e-5,
    "batch_size": 4,               # raw EEG waveforms → large tensors
    "gradient_accumulation_steps": 4,
    "early_stopping_patience": 8,
    "weight_decay": 1e-4,
    "max_grad_norm": 1.0,

    # ── Embedding ─────────────────────────────────────────────────────────────
    "embedding_dim": 128,          # final output embedding dimension

    # ── Data split ────────────────────────────────────────────────────────────
    "val_frac": 0.10,
    "test_frac": 0.10,
    "seed": 42,

    # ── Output files ──────────────────────────────────────────────────────────
    "shap_output": "shap_model_3.png",
    "pred_output": "prediction_model_3.png",
    "checkpoint_msm": "ckpt_model_3_msm.pt",
    "checkpoint_sup": "ckpt_model_3_sup.pt",
}

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. Logging setup
# ─────────────────────────────────────────────────────────────────────────────
def setup_logging(log_file: str) -> logging.Logger:
    """Configure root logger with both file and console handlers.

    Parameters
    ----------
    log_file : str
        Path to the log file.

    Returns
    -------
    logging.Logger
        Configured logger.
    """
    fmt = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    logger = logging.getLogger("model_3")
    logger.setLevel(logging.DEBUG)

    # Console handler
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)
    logger.addHandler(ch)

    # File handler
    fh = logging.FileHandler(log_file, mode="w")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    return logger

log = setup_logging(CONFIG["log_file"])

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4a. Clinical metadata loader
# ─────────────────────────────────────────────────────────────────────────────
def load_clinical_metadata(config: Dict) -> pd.DataFrame:
    """Load per-case clinical metadata from VitalDB.

    Parameters
    ----------
    config : Dict
        Global CONFIG dictionary (uses 'clinical_params').

    Returns
    -------
    pd.DataFrame
        One row per case with clinical columns.
    """
    log.info("Loading clinical metadata …")
    caseids = list(range(1, 6389))
    try:
        df = vitaldb.load_clinical_data(
            caseids=caseids,
            params=config["clinical_params"],
        )
    except Exception as exc:
        log.error(f"load_clinical_data failed: {exc}")
        raise
    log.info(f"Clinical metadata loaded: {len(df)} rows, {list(df.columns)}")
    return df

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4b. Waveform data loader (EEG-only tracks)
#     Loads EEG at 128 Hz and BIS scalars at 1 Hz.
# ─────────────────────────────────────────────────────────────────────────────
def load_waveform_data(
    caseids: List[int],
    config: Dict,
) -> Tuple[Dict[int, np.ndarray], Dict[int, np.ndarray]]:
    """Load raw EEG waveforms (128 Hz) and BIS scalars (1 Hz) for valid cases.

    Parameters
    ----------
    caseids : List[int]
        List of case IDs to attempt loading.
    config : Dict
        Global CONFIG dictionary.

    Returns
    -------
    wav_data : Dict[int, np.ndarray]
        Mapping caseid → float32 array (T_wav, 2) at 128 Hz.
    scalar_data : Dict[int, np.ndarray]
        Mapping caseid → float32 array (T_1hz, n_scalar_tracks) at 1 Hz.
    """
    wav_data: Dict[int, np.ndarray] = {}
    scalar_data: Dict[int, np.ndarray] = {}

    wav_tracks    = config["wav_tracks"]          # BIS/EEG1_WAV, BIS/EEG2_WAV
    scalar_tracks = config["scalar_tracks"]       # BIS/BIS, SQI, …, ART_MBP, HR
    wav_interval    = config["wav_interval"]      # 1/128
    scalar_interval = config["scalar_interval"]   # 1

    for caseid in tqdm(caseids, desc="Loading EEG waveforms"):

        # ── Load 128 Hz waveforms ─────────────────────────────────────────────
        try:
            arr_wav = vitaldb.load_case(
                caseid=caseid,
                track_names=wav_tracks,
                interval=wav_interval,
            )
        except Exception as exc:
            log.warning(f"caseid {caseid}: wav load failed — {exc}")
            arr_wav = None

        if arr_wav is None:
            log.warning(f"caseid {caseid}: wav returned None, skipping")
            continue

        # ── Load 1 Hz scalars ─────────────────────────────────────────────────
        try:
            arr_scalar = vitaldb.load_case(
                caseid=caseid,
                track_names=scalar_tracks,
                interval=scalar_interval,
            )
        except Exception as exc:
            log.warning(f"caseid {caseid}: scalar load failed — {exc}")
            del arr_wav
            gc.collect()
            continue

        if arr_scalar is None:
            log.warning(f"caseid {caseid}: scalar returned None, skipping")
            del arr_wav
            gc.collect()
            continue

        wav_data[caseid]    = arr_wav.astype(np.float32)
        scalar_data[caseid] = arr_scalar.astype(np.float32)

    log.info(
        f"Loaded waveform data for {len(wav_data)} cases "
        f"({len(caseids) - len(wav_data)} skipped)"
    )
    return wav_data, scalar_data

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Utility: max consecutive run of 1s in a binary array
# ─────────────────────────────────────────────────────────────────────────────
def max_run_length(arr: np.ndarray) -> int:
    """Return the maximum number of consecutive 1s in a binary array.

    Parameters
    ----------
    arr : np.ndarray
        1D binary integer array.

    Returns
    -------
    int
        Maximum run length of consecutive 1s.
    """
    if arr.sum() == 0:
        return 0
    max_run = 0
    current = 0
    for v in arr:
        if v:
            current += 1
            max_run = max(max_run, current)
        else:
            current = 0
    return max_run

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4c. IOH label construction
# ─────────────────────────────────────────────────────────────────────────────
def build_ioh_labels(
    map_series: np.ndarray,
    config: Dict,
) -> Dict[str, np.ndarray]:
    """Build binary IOH labels at each prediction horizon.

    A positive label is assigned when MAP drops below threshold and stays below
    for at least min_duration_sec within the prediction window.

    Parameters
    ----------
    map_series : np.ndarray
        1D float array of MAP values at 1 Hz.
    config : Dict
        Global CONFIG dictionary.

    Returns
    -------
    Dict[str, np.ndarray]
        Keys: 'ioh_5', 'ioh_10', 'ioh_15'; values: binary float32 1D arrays.
    """
    threshold = config["ioh_thresholds_mmhg"]
    min_dur   = config["ioh_min_duration_sec"]
    labels: Dict[str, np.ndarray] = {}

    for horizon_sec in config["ioh_horizons_sec"]:
        n   = len(map_series)
        lbl = np.zeros(n, dtype=np.float32)
        for t in range(n - horizon_sec):
            window = map_series[t: t + horizon_sec]
            below  = (window < threshold).astype(int)
            run    = max_run_length(below)
            if run >= min_dur:
                lbl[t] = 1.0
        key = f"ioh_{horizon_sec // 60}"  # e.g. 'ioh_5'
        labels[key] = lbl

    return labels

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4d. N/H label construction (noisy supervision)
# ─────────────────────────────────────────────────────────────────────────────
def build_nh_labels(
    bis_series: np.ndarray,
    hr_series: np.ndarray,
    config: Dict,
) -> np.ndarray:
    """Build 3-class N/H labels from heuristic rules (noisy supervision).

    Classes
    -------
    0 : Adequate balance
    1 : Nociception excess  — concurrent HR > 90 bpm AND BIS < 60
    2 : Hypnosis excess     — rising BIS > 65 AND HR rise > 15 % over 5 min

    Parameters
    ----------
    bis_series : np.ndarray
        1D BIS values at 1 Hz.
    hr_series : np.ndarray
        1D HR values at 1 Hz.
    config : Dict
        Global CONFIG dict.

    Returns
    -------
    np.ndarray
        Integer array of class indices (0, 1, 2), shape (T,).
    """
    n      = len(bis_series)
    labels = np.zeros(n, dtype=np.int64)

    hr_high      = config["nh_hr_high"]
    bis_low      = config["nh_bis_low"]
    bis_high     = config["nh_bis_high"]
    hr_rise_frac = config["nh_hr_rise_frac"]
    lookback     = 300  # 5-min lookback for HR rise calculation

    for t in range(lookback, n):
        bis_t = bis_series[t]
        hr_t  = hr_series[t]
        if np.isnan(bis_t) or np.isnan(hr_t):
            continue

        # Nociception excess: high HR AND low BIS simultaneously
        if hr_t > hr_high and bis_t < bis_low:
            labels[t] = 1
            continue

        # Hypnosis excess: BIS rising above threshold AND HR increased ≥ 15 %
        hr_past = hr_series[t - lookback: t]
        if not np.all(np.isnan(hr_past)):
            hr_mean_past = float(np.nanmean(hr_past))
            if hr_mean_past > 0:
                hr_rise = (hr_t - hr_mean_past) / hr_mean_past
                if bis_t > bis_high and hr_rise > hr_rise_frac:
                    labels[t] = 2

    return labels

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4e. Emergence timing label (stored but not primary for Model 3)
# ─────────────────────────────────────────────────────────────────────────────
def build_et_label(row: pd.Series) -> float:
    """Return emergence time (aneend - opend) in seconds, or -1 if invalid.

    Parameters
    ----------
    row : pd.Series
        A single row from the clinical metadata DataFrame.

    Returns
    -------
    float
        Emergence duration in seconds, or -1.0 if outside valid range.
    """
    try:
        et = float(row["aneend"]) - float(row["opend"])
    except (TypeError, ValueError):
        return -1.0
    # Keep only 0 < ET < 3600 s (per spec)
    if et <= 0 or et >= 3600:
        return -1.0
    return float(et)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5a. Forward-fill / backward-fill NaN gaps
# ─────────────────────────────────────────────────────────────────────────────
def forward_fill_gaps(arr: np.ndarray, max_gap_sec: int) -> np.ndarray:
    """Forward-fill NaN gaps up to max_gap_sec; backward-fill leading NaNs.

    Parameters
    ----------
    arr : np.ndarray
        2D float array (T, F). Operates column-wise.
    max_gap_sec : int
        Maximum consecutive NaN samples to fill (1 Hz → 1 sample per second).

    Returns
    -------
    np.ndarray
        Filled array, same shape as input.
    """
    out = arr.copy()
    T, F = out.shape

    for f in range(F):
        col     = out[:, f]
        nan_idx = np.where(np.isnan(col))[0]
        if len(nan_idx) == 0:
            continue

        gaps = np.split(nan_idx, np.where(np.diff(nan_idx) != 1)[0] + 1)
        for gap in gaps:
            gap_len = len(gap)
            start   = gap[0]
            end     = gap[-1]

            if gap_len > max_gap_sec:
                continue  # too long — leave as NaN for window discard

            if start > 0 and not np.isnan(col[start - 1]):
                col[gap] = col[start - 1]
            elif end < T - 1 and not np.isnan(col[end + 1]):
                col[gap] = col[end + 1]

        out[:, f] = col

    return out

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5b. Outlier removal (physiological clipping for scalar channels)
# ─────────────────────────────────────────────────────────────────────────────
def remove_outliers(arr: np.ndarray, method: str = "physiological") -> np.ndarray:
    """Clip physiologically implausible values in scalar channels.

    Out-of-range values are set to NaN (handled by forward-fill or window discard).

    Parameters
    ----------
    arr : np.ndarray
        2D float array (T, F) — must correspond to CONFIG['scalar_tracks'] order.
    method : str
        'physiological' (default) or 'iqr'.

    Returns
    -------
    np.ndarray
        Clipped array, same shape.
    """
    out = arr.copy()

    if method == "physiological":
        # Column order matches CONFIG['scalar_tracks']:
        # 0:BIS/BIS  1:BIS/SQI  2:BIS/EMG  3:BIS/SR  4:BIS/SEF  5:BIS/TOTPOW
        # 6:Solar8000/ART_MBP  7:Solar8000/HR
        bounds = [
            (0, 100),      # BIS
            (0, 100),      # SQI
            (0, 100),      # EMG
            (0, 100),      # SR (burst suppression ratio 0–100 %)
            (0, 50),       # SEF (spectral edge frequency Hz)
            (0, 200),      # TOTPOW (dB arbitrary units)
            (20, 200),     # MAP mmHg
            (20, 250),     # HR bpm
        ]
        n_cols = min(out.shape[1], len(bounds))
        for col_idx in range(n_cols):
            lo, hi = bounds[col_idx]
            col = out[:, col_idx]
            col = np.where((col < lo) | (col > hi), np.nan, col)
            out[:, col_idx] = col

    elif method == "iqr":
        for col_idx in range(out.shape[1]):
            col   = out[:, col_idx]
            valid = col[~np.isnan(col)]
            if len(valid) < 4:
                continue
            q1, q3 = np.percentile(valid, [25, 75])
            iqr     = q3 - q1
            lo, hi  = q1 - 3.0 * iqr, q3 + 3.0 * iqr
            out[:, col_idx] = np.where((col < lo) | (col > hi), np.nan, col)

    return out

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5c. Spectral feature computation (EEG-specific, 128 Hz input)
# ─────────────────────────────────────────────────────────────────────────────
def compute_spectral_features(
    eeg_window: np.ndarray,
    config: Dict,
) -> np.ndarray:
    """Compute band powers, SEF95, and BSR from a raw EEG segment.

    Operates on 128 Hz EEG. The STFT window is 4 s (512 samples at 128 Hz).
    Maximum resolvable frequency is 64 Hz (Nyquist), which comfortably covers
    all clinical EEG bands (delta–beta).

    Parameters
    ----------
    eeg_window : np.ndarray
        1D float array (N_samples,) at 128 Hz.
    config : Dict
        Global CONFIG dict.

    Returns
    -------
    np.ndarray
        1D feature vector: [delta_power, theta_power, alpha_power, beta_power,
                             sef95, bsr] — shape (6,).
    """
    fs       = config["eeg_fs"]          # 128 Hz
    win_sec  = config["stft_win_sec"]    # 4 s → 512 samples
    overlap  = config["stft_overlap"]   # 50 %
    bands    = config["stft_bands"]
    bsr_win  = config["bsr_window_sec"] # 63 s
    bsr_amp  = config["bsr_amplitude_uv"]

    nperseg  = int(win_sec * fs)         # 512 samples
    noverlap = int(nperseg * overlap)    # 256 samples

    try:
        freqs, _, Zxx = scipy_stft(
            eeg_window, fs=fs,
            window="hann", nperseg=nperseg, noverlap=noverlap,
        )
        power      = np.abs(Zxx) ** 2         # (n_freqs, n_frames)
        mean_power = power.mean(axis=1)       # average over time frames

        band_powers = []
        for band_name, (flo, fhi) in bands.items():
            mask = (freqs >= flo) & (freqs < fhi)
            bp   = mean_power[mask].sum() if mask.any() else 0.0
            band_powers.append(float(bp))

        # SEF95: frequency below which 95 % of total power resides
        total_power = mean_power.sum()
        if total_power > 0:
            cum_power = np.cumsum(mean_power)
            sef95_idx = np.searchsorted(cum_power, 0.95 * total_power)
            sef95     = float(freqs[min(sef95_idx, len(freqs) - 1)])
        else:
            sef95 = 0.0

    except Exception as exc:
        log.debug(f"STFT failed: {exc}")
        band_powers = [0.0, 0.0, 0.0, 0.0]
        sef95       = 0.0

    # BSR: fraction of 63-s window with amplitude < bsr_amp µV
    bsr_samples = int(bsr_win * fs)   # 63 × 128 = 8064 samples
    segment     = eeg_window[-bsr_samples:] if len(eeg_window) >= bsr_samples else eeg_window
    bsr         = float((np.abs(segment) < bsr_amp).mean()) if len(segment) > 0 else 0.0

    return np.array(band_powers + [sef95, bsr], dtype=np.float32)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. Window generation
#    eeg_samples_per_step = 3840 (30 s × 128 Hz)
# ─────────────────────────────────────────────────────────────────────────────
def generate_windows(
    wav_data: Dict[int, np.ndarray],
    scalar_data: Dict[int, np.ndarray],
    clinical_df: pd.DataFrame,
    config: Dict,
) -> List[Dict]:
    """Extract labelled sliding windows from all valid cases.

    Each window contains:
    - Raw EEG waveform chunk (3840, 2) for the 30-s preceding each 1-Hz step
    - Scalar features for the 1800-s history
    - Pre-computed spectral features per 1-Hz step
    - IOH, N/H, ET labels

    Parameters
    ----------
    wav_data : Dict[int, np.ndarray]
        Waveform arrays at 128 Hz.
    scalar_data : Dict[int, np.ndarray]
        Scalar arrays at 1 Hz.
    clinical_df : pd.DataFrame
        Clinical metadata.
    config : Dict
        Global CONFIG dict.

    Returns
    -------
    List[Dict]
    """
    windows: List[Dict] = []

    history_sec          = config["history_sec"]
    stride_sec           = config["stride_sec"]
    max_nan_frac         = config["max_nan_frac"]
    eeg_samples_per_step = config["eeg_samples_per_step"]  # 3840
    eeg_fs               = config["eeg_fs"]                # 128

    sqi_col_idx = 1   # BIS/SQI
    map_col_idx = 6   # Solar8000/ART_MBP
    hr_col_idx  = 7   # Solar8000/HR
    bis_col_idx = 0   # BIS/BIS

    clinical_df = clinical_df.set_index("caseid")

    for caseid, arr_scalar in tqdm(scalar_data.items(), desc="Generating windows"):
        arr_wav = wav_data.get(caseid)
        if arr_wav is None:
            log.warning(f"caseid {caseid}: no waveform data, skipping window generation")
            continue

        if caseid not in clinical_df.index:
            log.warning(f"caseid {caseid}: missing clinical metadata, skipping")
            del arr_wav; gc.collect(); continue

        row      = clinical_df.loc[caseid]
        T_scalar = arr_scalar.shape[0]

        if T_scalar < config["min_case_duration_sec"]:
            log.debug(f"caseid {caseid}: too short ({T_scalar}s), skipping")
            del arr_wav; gc.collect(); continue

        # ── SQI coverage check ────────────────────────────────────────────────
        sqi_col  = arr_scalar[:, sqi_col_idx]
        valid_sqi = ~np.isnan(sqi_col)
        if valid_sqi.sum() > 0:
            sqi_coverage = (sqi_col[valid_sqi] >= config["sqi_train_threshold"]).mean()
        else:
            sqi_coverage = 0.0

        if sqi_coverage < config["min_sqi_coverage"]:
            log.debug(f"caseid {caseid}: SQI coverage {sqi_coverage:.2f} < threshold, skipping")
            del arr_wav; gc.collect(); continue

        # ── Pre-process scalar array ──────────────────────────────────────────
        arr_scalar = remove_outliers(arr_scalar, method="physiological")
        arr_scalar = forward_fill_gaps(arr_scalar, max_gap_sec=config["max_fill_gap_sec"])

        # ── Build labels ──────────────────────────────────────────────────────
        map_series = arr_scalar[:, map_col_idx]
        ioh_labels = build_ioh_labels(map_series, config)
        bis_series = arr_scalar[:, bis_col_idx]
        hr_series  = arr_scalar[:, hr_col_idx]
        nh_labels  = build_nh_labels(bis_series, hr_series, config)
        et_val     = build_et_label(row)

        has_ioh_positive = any(lbl.sum() > 0 for lbl in ioh_labels.values())
        has_ioh_negative = any((1 - lbl).sum() > 0 for lbl in ioh_labels.values())
        if not (has_ioh_positive or has_ioh_negative):
            log.debug(f"caseid {caseid}: no valid IOH labels, skipping")
            del arr_wav; gc.collect(); continue

        # ── Spectral features per 1-Hz timestep ──────────────────────────────
        # Use the preceding 30 s of EEG1 (3840 samples at 128 Hz).
        eeg_ch0         = arr_wav[:, 0]  # EEG1_WAV channel
        n_steps         = T_scalar
        spectral_features = np.zeros((n_steps, 6), dtype=np.float32)

        for t in range(n_steps):
            wav_t_end   = int((t + 1) * eeg_fs)   # end sample at 128 Hz
            wav_t_start = max(0, wav_t_end - eeg_samples_per_step)
            if wav_t_end > len(eeg_ch0):
                break
            seg = eeg_ch0[wav_t_start:wav_t_end]
            spectral_features[t] = compute_spectral_features(seg, config)

        # ── Sliding window extraction ─────────────────────────────────────────
        case_windows_before = len(windows)

        for t_end in range(history_sec, T_scalar, stride_sec):
            t_start    = t_end - history_sec
            win_scalar = arr_scalar[t_start:t_end, :]

            # Discard window if any channel exceeds NaN threshold
            discard = any(
                np.isnan(win_scalar[:, f_idx]).mean() > max_nan_frac
                for f_idx in range(win_scalar.shape[1])
            )
            if discard:
                continue

            win_scalar = np.nan_to_num(win_scalar, nan=0.0)

            # Raw EEG for the last 30 s: shape (3840, 2)
            wav_end_sample   = int(t_end * eeg_fs)
            wav_start_sample = max(0, wav_end_sample - eeg_samples_per_step)
            if wav_end_sample > arr_wav.shape[0]:
                continue
            win_wav = arr_wav[wav_start_sample:wav_end_sample, :]
            if win_wav.shape[0] < eeg_samples_per_step:
                continue

            win_spectral = spectral_features[t_start:t_end, :]   # (1800, 6)

            # SQI gate
            sqi_at_end       = arr_scalar[t_end - 1, sqi_col_idx]
            modality_present = not np.isnan(sqi_at_end)

            sqi_win  = arr_scalar[t_start:t_end, sqi_col_idx]
            sqi_mask = (
                ~np.isnan(sqi_win) & (sqi_win >= config["sqi_train_threshold"])
            ).astype(np.float32)

            # Labels at prediction point t_end
            label_ioh_5  = float(ioh_labels["ioh_5"][t_end])  if t_end < len(ioh_labels["ioh_5"])  else 0.0
            label_ioh_10 = float(ioh_labels["ioh_10"][t_end]) if t_end < len(ioh_labels["ioh_10"]) else 0.0
            label_ioh_15 = float(ioh_labels["ioh_15"][t_end]) if t_end < len(ioh_labels["ioh_15"]) else 0.0
            label_nh     = int(nh_labels[t_end]) if t_end < len(nh_labels) else 0

            windows.append({
                "caseid":           caseid,
                "window_start":     t_start,
                "win_scalar":       win_scalar.astype(np.float32),      # (1800, 8)
                "win_wav":          win_wav.astype(np.float32),          # (3840, 2)
                "win_spectral":     win_spectral.astype(np.float32),    # (1800, 6)
                "sqi_mask":         sqi_mask,                            # (1800,)
                "label_ioh_5":      label_ioh_5,
                "label_ioh_10":     label_ioh_10,
                "label_ioh_15":     label_ioh_15,
                "label_nh":         label_nh,
                "label_et":         et_val,
                "modality_present": modality_present,
            })

        case_n_windows = len(windows) - case_windows_before
        log.debug(f"caseid {caseid}: {case_n_windows} windows generated")

        del arr_wav
        del wav_data[caseid]
        gc.collect()

    log.info(f"Total windows generated: {len(windows)}")
    return windows

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. Scaler fitting and application
# ─────────────────────────────────────────────────────────────────────────────
def fit_scalers(
    train_windows: List[Dict],
) -> Tuple[StandardScaler, StandardScaler]:
    """Fit StandardScalers on training data only.

    Fitted on training split only — never on val/test.

    Parameters
    ----------
    train_windows : List[Dict]

    Returns
    -------
    scaler_scalar : StandardScaler
    scaler_spectral : StandardScaler
    """
    scalar_stack = np.concatenate(
        [w["win_scalar"].reshape(-1, w["win_scalar"].shape[-1]) for w in train_windows], axis=0
    )
    spectral_stack = np.concatenate(
        [w["win_spectral"].reshape(-1, w["win_spectral"].shape[-1]) for w in train_windows], axis=0
    )
    scaler_scalar   = StandardScaler().fit(scalar_stack)
    scaler_spectral = StandardScaler().fit(spectral_stack)
    log.info("Scalers fitted on training data.")
    return scaler_scalar, scaler_spectral


def apply_scalers(
    windows: List[Dict],
    scaler_scalar: StandardScaler,
    scaler_spectral: StandardScaler,
) -> List[Dict]:
    """Apply pre-fitted scalers to a list of window dicts in-place.

    Parameters
    ----------
    windows : List[Dict]
    scaler_scalar : StandardScaler
    scaler_spectral : StandardScaler

    Returns
    -------
    List[Dict]
    """
    for w in windows:
        T,  F  = w["win_scalar"].shape
        T2, F2 = w["win_spectral"].shape
        w["win_scalar"] = (
            scaler_scalar.transform(w["win_scalar"].reshape(-1, F))
            .reshape(T, F).astype(np.float32)
        )
        w["win_spectral"] = (
            scaler_spectral.transform(w["win_spectral"].reshape(-1, F2))
            .reshape(T2, F2).astype(np.float32)
        )
    return windows

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 8. Train / Val / Test split (patient-level)
# ─────────────────────────────────────────────────────────────────────────────
def split_cases(
    all_caseids: List[int],
    val_frac: float = 0.10,
    test_frac: float = 0.10,
    seed: int = 42,
) -> Tuple[List[int], List[int], List[int]]:
    """Patient-level split — never split one patient across sets.

    Returns
    -------
    Tuple[List[int], List[int], List[int]]
        (train_caseids, val_caseids, test_caseids)
    """
    train_val, test = train_test_split(all_caseids, test_size=test_frac, random_state=seed)
    val_size        = val_frac / (1.0 - test_frac)
    train, val      = train_test_split(train_val, test_size=val_size, random_state=seed)
    log.info(f"Split: train={len(train)}, val={len(val)}, test={len(test)} cases")
    return train, val, test

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 9. EEGDataset — PyTorch Dataset
# ─────────────────────────────────────────────────────────────────────────────
class EEGDataset(Dataset):
    """PyTorch Dataset for Model 3 EEG windows.

    Parameters
    ----------
    windows : List[Dict]
        Pre-processed window dicts.
    """

    def __init__(self, windows: List[Dict]) -> None:
        self.windows = windows

    def __len__(self) -> int:
        return len(self.windows)

    def __getitem__(self, idx: int) -> Dict:
        """Return tensors for one window.

        Returns
        -------
        Dict with keys:
            x                : FloatTensor (T=1800, n_scalar+n_spectral=14)
            wav              : FloatTensor (3840, 2)  — raw EEG at 128 Hz
            sqi_flag         : FloatTensor (T,)       — 1.0 where SQI >= 70
            label_ioh_5/10/15: FloatTensor scalar
            label_nh         : LongTensor scalar class index (0/1/2)
            label_et         : FloatTensor scalar (-1 if unavailable)
            caseid           : int
            window_start     : int
            modality_present : BoolTensor scalar
        """
        w = self.windows[idx]
        # Concatenate scalar (8) and spectral (6) features → (1800, 14)
        x = np.concatenate([w["win_scalar"], w["win_spectral"]], axis=-1)
        return {
            "x":                torch.FloatTensor(x),
            "wav":              torch.FloatTensor(w["win_wav"]),
            "sqi_flag":         torch.FloatTensor(w["sqi_mask"]),
            "label_ioh_5":      torch.FloatTensor([w["label_ioh_5"]]),
            "label_ioh_10":     torch.FloatTensor([w["label_ioh_10"]]),
            "label_ioh_15":     torch.FloatTensor([w["label_ioh_15"]]),
            "label_nh":         torch.LongTensor([w["label_nh"]]),
            "label_et":         torch.FloatTensor([w["label_et"]]),
            "caseid":           w["caseid"],
            "window_start":     w["window_start"],
            "modality_present": torch.BoolTensor([w["modality_present"]]),
        }

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 10. DataLoaders
# ─────────────────────────────────────────────────────────────────────────────
def build_dataloaders(
    train_ds: EEGDataset,
    val_ds:   EEGDataset,
    test_ds:  EEGDataset,
    config:   Dict,
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """Build train / val / test DataLoaders."""
    bs = config["batch_size"]
    # num_workers=0 — raw EEG tensors are large; multiprocessing adds overhead
    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,
                              num_workers=0, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=bs, shuffle=False,
                              num_workers=0, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=bs, shuffle=False,
                              num_workers=0, pin_memory=True)
    log.info(
        f"DataLoaders: train={len(train_loader)} batches, "
        f"val={len(val_loader)}, test={len(test_loader)}"
    )
    return train_loader, val_loader, test_loader

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 11. Model architecture
# ─────────────────────────────────────────────────────────────────────────────

class CausalConv1d(nn.Module):
    """1D causal convolution: left-pads so output[t] depends only on input[<=t]."""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        dilation: int = 1,
    ) -> None:
        super().__init__()
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels, out_channels,
            kernel_size=kernel_size, dilation=dilation, padding=0,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(F.pad(x, (self.padding, 0)))


class WaveNetLayer(nn.Module):
    """Single WaveNet residual layer: dilated causal conv + gated activation + skip."""

    def __init__(
        self,
        residual_channels: int,
        skip_channels: int,
        kernel_size: int = 3,
        dilation: int = 1,
    ) -> None:
        super().__init__()
        self.causal_conv = CausalConv1d(
            residual_channels, 2 * residual_channels,
            kernel_size=kernel_size, dilation=dilation,
        )
        self.res_conv  = nn.Conv1d(residual_channels, residual_channels, kernel_size=1)
        self.skip_conv = nn.Conv1d(residual_channels, skip_channels,     kernel_size=1)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h = self.causal_conv(x)
        C = h.size(1) // 2
        h = torch.tanh(h[:, :C, :]) * torch.sigmoid(h[:, C:, :])
        skip = self.skip_conv(h)
        res  = self.res_conv(h) + x
        return res, skip


class WaveNetEncoder(nn.Module):
    """16-layer WaveNet dilated causal CNN.

    Compresses (B, 2, 3840) raw EEG (128 Hz) to (B, skip_channels, n_output_vectors)
    via adaptive average pooling.
    """

    def __init__(
        self,
        n_input_channels: int = 2,
        residual_channels: int = 64,
        skip_channels: int = 64,
        kernel_size: int = 3,
        dilations: Optional[List[int]] = None,
        n_output_vectors: int = 256,
    ) -> None:
        super().__init__()
        if dilations is None:
            dilations = [1, 2, 4, 8, 16, 32, 64, 128, 1, 2, 4, 8, 16, 32, 64, 128]

        self.input_proj = nn.Conv1d(n_input_channels, residual_channels, kernel_size=1)
        self.layers = nn.ModuleList([
            WaveNetLayer(residual_channels, skip_channels, kernel_size, d)
            for d in dilations
        ])
        self.post_skip = nn.Sequential(
            nn.ReLU(),
            nn.Conv1d(skip_channels, skip_channels, kernel_size=1),
            nn.ReLU(),
        )
        self.adaptive_pool = nn.AdaptiveAvgPool1d(n_output_vectors)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : Tensor (B, 2, 3840)

        Returns
        -------
        Tensor (B, skip_channels, n_output_vectors)
        """
        h        = self.input_proj(x)
        skip_sum = None
        for layer in self.layers:
            h, skip  = layer(h)
            skip_sum = skip if skip_sum is None else skip_sum + skip
        out = self.post_skip(skip_sum)
        out = self.adaptive_pool(out)
        return out

In [ ]:
class MSMDecoder(nn.Module):
    """Decoder head for Masked Segment Modelling pretraining.

    Reconstructs masked raw EEG segments (128 Hz) from CNN feature maps.
    """

    def __init__(
        self,
        cnn_out_dim: int,
        n_output_samples: int = 3840,   # 30 s × 128 Hz
        n_eeg_channels: int = 2,
    ) -> None:
        super().__init__()
        self.n_output_samples = n_output_samples
        self.n_eeg_channels   = n_eeg_channels

        # Upsample feature map back to raw waveform length
        self.upsample = nn.Sequential(
            nn.ConvTranspose1d(cnn_out_dim, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose1d(32, 16, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose1d(16, n_eeg_channels, kernel_size=4, stride=2, padding=1),
        )
        self.adaptive_upsample = nn.AdaptiveAvgPool1d(n_output_samples)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        z : Tensor (B, cnn_out_dim, n_output_vectors)

        Returns
        -------
        Tensor (B, n_eeg_channels, n_output_samples)  e.g. (B, 2, 3840)
        """
        x = self.upsample(z)
        x = self.adaptive_upsample(x)
        return x


class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding for Transformer encoder."""

    def __init__(self, d_model: int, max_len: int = 600, dropout: float = 0.1) -> None:
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, : x.size(1), :]
        return self.dropout(x)


class EEGTransformerEncoder(nn.Module):
    """4-layer Transformer encoder over WaveNet features + spectral features."""

    def __init__(
        self,
        cnn_out_dim: int = 64,
        n_spectral_features: int = 9,
        d_model: int = 128,
        n_heads: int = 4,
        n_layers: int = 4,
        ffn_dim: int = 256,
        dropout: float = 0.1,
        n_output_vectors: int = 256,
    ) -> None:
        super().__init__()
        self.cnn_proj      = nn.Linear(cnn_out_dim, d_model)
        self.spectral_proj = nn.Linear(n_spectral_features, d_model)
        self.spectral_pool = nn.AdaptiveAvgPool1d(n_output_vectors)
        self.pos_enc       = PositionalEncoding(d_model, max_len=n_output_vectors, dropout=dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=ffn_dim, dropout=dropout, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.out_proj    = nn.Linear(d_model, d_model)

    def forward(
        self,
        cnn_features:      torch.Tensor,
        spectral_features: torch.Tensor,
        sqi_mask:          Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Parameters
        ----------
        cnn_features      : Tensor (B, cnn_out_dim, n_output_vectors)
        spectral_features : Tensor (B, T_scalar, n_spectral_features)
        sqi_mask          : Tensor (B, T_scalar)  optional

        Returns
        -------
        embedding : Tensor (B, 128)
        """
        # CNN features → sequence of d_model vectors
        cnn_seq = self.cnn_proj(cnn_features.permute(0, 2, 1))     # (B, N_vec, d_model)

        # Spectral features → pool to match N_vec
        sp_proj = self.spectral_proj(spectral_features)             # (B, T_scalar, d_model)
        sp_proj = self.spectral_pool(sp_proj.permute(0, 2, 1))     # (B, d_model, N_vec)
        sp_proj = sp_proj.permute(0, 2, 1)                         # (B, N_vec, d_model)

        fused   = self.pos_enc(cnn_seq + sp_proj)
        encoded = self.transformer(fused)                           # (B, N_vec, d_model)
        embedding = self.out_proj(encoded.mean(dim=1))             # (B, d_model)
        return embedding

In [ ]:
class EEGModel(nn.Module):
    """Full EEG Model 3: WaveNet CNN → Transformer + task heads.

    Architecture
    ------------
    1. WaveNetEncoder  : raw EEG (B, 2, 3840) @ 128 Hz → CNN features (B, 64, 256)
    2. EEGTransformerEncoder : CNN + spectral features → 128-dim embedding
    3. IOH head        : binary classification for 3 horizons (5, 10, 15 min)
    4. N/H head        : 3-class classification (adequate / nociception / hypnosis)

    Stage 0 (MSM pretraining) uses only the WaveNet encoder + MSM decoder.
    """

    def __init__(self, config: Dict) -> None:
        super().__init__()
        self.config = config

        self.wavenet = WaveNetEncoder(
            n_input_channels=2,
            residual_channels=config["wavenet_residual_channels"],
            skip_channels=config["wavenet_skip_channels"],
            kernel_size=config["wavenet_kernel_size"],
            dilations=config["wavenet_dilations"],
            n_output_vectors=config["wavenet_n_output_vectors"],
        )
        self.msm_decoder = MSMDecoder(
            cnn_out_dim=config["wavenet_skip_channels"],
            n_output_samples=config["eeg_samples_per_step"],  # 3840
            n_eeg_channels=2,
        )
        self.transformer_enc = EEGTransformerEncoder(
            cnn_out_dim=config["wavenet_skip_channels"],
            n_spectral_features=config["transformer_n_spectral_features"],
            d_model=config["transformer_d_model"],
            n_heads=config["transformer_n_heads"],
            n_layers=config["transformer_n_layers"],
            ffn_dim=config["transformer_ffn_dim"],
            dropout=config["transformer_dropout"],
            n_output_vectors=config["wavenet_n_output_vectors"],
        )
        emb_dim = config["embedding_dim"]
        self.ioh_head_5  = nn.Linear(emb_dim, 1)
        self.ioh_head_10 = nn.Linear(emb_dim, 1)
        self.ioh_head_15 = nn.Linear(emb_dim, 1)
        self.nh_head     = nn.Linear(emb_dim, config["nh_n_classes"])

    def encode(
        self,
        wav:         torch.Tensor,
        x_spectral:  torch.Tensor,
        sqi_mask:    Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """Run WaveNet + Transformer to produce 128-dim embedding.

        Parameters
        ----------
        wav        : Tensor (B, 2, 3840)
        x_spectral : Tensor (B, T, n_spectral_features)
        sqi_mask   : Tensor (B, T) optional

        Returns
        -------
        Tensor (B, 128)
        """
        cnn_feat  = self.wavenet(wav)
        embedding = self.transformer_enc(cnn_feat, x_spectral, sqi_mask)
        return embedding

    def forward(
        self,
        wav:              torch.Tensor,
        x_spectral:       torch.Tensor,
        sqi_mask:         Optional[torch.Tensor] = None,
        modality_present: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        """Full forward pass for supervised training.

        Returns
        -------
        Dict with keys: 'embedding', 'ioh_5', 'ioh_10', 'ioh_15', 'nh'
        """
        embedding = self.encode(wav, x_spectral, sqi_mask)

        # Zero-out embedding where SQI < 50 (modality_present = False)
        if modality_present is not None:
            embedding = embedding * modality_present.float().unsqueeze(-1)

        return {
            "embedding": embedding,
            "ioh_5":     self.ioh_head_5(embedding),
            "ioh_10":    self.ioh_head_10(embedding),
            "ioh_15":    self.ioh_head_15(embedding),
            "nh":        self.nh_head(embedding),
        }

    def forward_msm(self, wav: torch.Tensor) -> torch.Tensor:
        """MSM pretraining forward pass.

        Parameters
        ----------
        wav : Tensor (B, 2, 3840) — partially masked raw EEG at 128 Hz

        Returns
        -------
        recon : Tensor (B, 2, 3840)
        """
        cnn_feat = self.wavenet(wav)
        return self.msm_decoder(cnn_feat)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 12. Loss functions
# ─────────────────────────────────────────────────────────────────────────────
def ioh_bce_loss(
    pred:     torch.Tensor,
    target:   torch.Tensor,
    sqi_mask: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """Binary cross-entropy loss for IOH prediction, gated by SQI mask."""
    bce = F.binary_cross_entropy_with_logits(pred, target, reduction="none")
    if sqi_mask is not None:
        gate    = sqi_mask.float().unsqueeze(-1)
        bce     = bce * gate
        n_valid = gate.sum().clamp(min=1.0)
        return bce.sum() / n_valid
    return bce.mean()


def focal_loss(
    logits:          torch.Tensor,
    targets:         torch.Tensor,
    gamma:           float = 2.0,
    class_weights:   Optional[torch.Tensor] = None,
    label_smoothing: float = 0.15,
) -> torch.Tensor:
    """Focal loss for N/H multi-class classification with label smoothing."""
    n_classes = logits.size(-1)
    with torch.no_grad():
        smoothed = torch.zeros_like(logits).scatter_(1, targets.unsqueeze(1), 1.0)
        smoothed = smoothed * (1 - label_smoothing) + label_smoothing / n_classes

    log_probs = F.log_softmax(logits, dim=-1)
    ce        = -(smoothed * log_probs).sum(dim=-1)

    probs  = torch.exp(log_probs)
    pt     = (probs * F.one_hot(targets, n_classes).float()).sum(dim=-1)
    loss   = (1.0 - pt).pow(gamma) * ce

    if class_weights is not None:
        loss = loss * class_weights[targets]
    return loss.mean()


def msm_reconstruction_loss(
    recon:    torch.Tensor,
    original: torch.Tensor,
    mask:     torch.Tensor,
) -> torch.Tensor:
    """MSE loss computed only over masked positions.

    Parameters
    ----------
    recon    : Tensor (B, 2, T)
    original : Tensor (B, 2, T)
    mask     : Tensor (B, T) — 1.0 at masked positions
    """
    mask_expanded = mask.unsqueeze(1).expand_as(recon)
    n_masked      = mask_expanded.sum().clamp(min=1.0)
    return ((recon - original) ** 2 * mask_expanded).sum() / n_masked


def compute_class_weights(labels_nh: List[int], n_classes: int = 3) -> torch.Tensor:
    """Compute inverse frequency class weights from training label list."""
    arr     = np.array(labels_nh, dtype=np.int64)
    counts  = np.maximum(np.bincount(arr, minlength=n_classes).astype(np.float32), 1.0)
    weights = 1.0 / counts
    weights = weights / weights.sum() * n_classes
    return torch.FloatTensor(weights)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 13. Metrics
# ─────────────────────────────────────────────────────────────────────────────
def compute_metrics(
    ioh_preds:   Dict[str, np.ndarray],
    ioh_targets: Dict[str, np.ndarray],
    nh_preds:    np.ndarray,
    nh_targets:  np.ndarray,
) -> Dict[str, float]:
    """Compute AUROC, AUPRC for IOH and balanced accuracy/macro-F1 for N/H."""
    from sklearn.metrics import (
        average_precision_score, balanced_accuracy_score,
        f1_score, roc_auc_score,
    )
    metrics: Dict[str, float] = {}
    for key in ["ioh_5", "ioh_10", "ioh_15"]:
        p, t = ioh_preds[key], ioh_targets[key]
        if len(np.unique(t)) < 2:
            metrics[f"auroc_{key}"] = float("nan")
            metrics[f"auprc_{key}"] = float("nan")
        else:
            metrics[f"auroc_{key}"] = float(roc_auc_score(t, p))
            metrics[f"auprc_{key}"] = float(average_precision_score(t, p))
    metrics["nh_balanced_acc"] = float(balanced_accuracy_score(nh_targets, nh_preds))
    metrics["nh_macro_f1"]     = float(f1_score(nh_targets, nh_preds, average="macro", zero_division=0))
    return metrics

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MSM masking helper
#   span = 64 samples = 0.5 s at 128 Hz
# ─────────────────────────────────────────────────────────────────────────────
def apply_msm_mask(
    wav:    torch.Tensor,
    config: Dict,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Randomly mask 20 % of raw EEG with contiguous spans of 0.5 s (64 samples @ 128 Hz).

    Parameters
    ----------
    wav    : Tensor (B, 2, T)
    config : Dict

    Returns
    -------
    wav_masked : Tensor (B, 2, T)
    mask       : Tensor (B, T) — 1.0 at masked positions
    """
    B, C, T = wav.shape
    mask        = torch.zeros(B, T, device=wav.device)
    span        = config["msm_mask_span_samples"]          # 64
    target_masked = int(config["msm_mask_prob"] * T)       # 20 % of T

    wav_masked = wav.clone()
    for b in range(B):
        n_masked, attempts = 0, 0
        while n_masked < target_masked and attempts < 1000:
            start = random.randint(0, T - span)
            mask[b, start: start + span] = 1.0
            wav_masked[b, :, start: start + span] = 0.0
            n_masked = int(mask[b].sum().item())
            attempts += 1

    return wav_masked, mask

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 14a. Stage 0 — MSM pretraining epoch
# ─────────────────────────────────────────────────────────────────────────────
def pretrain_one_epoch_msm(
    model:      EEGModel,
    loader:     DataLoader,
    optimizer:  torch.optim.Optimizer,
    amp_scaler: GradScaler,
    config:     Dict,
    device:     torch.device,
) -> float:
    """Run one MSM pretraining epoch.

    Returns
    -------
    float
        Mean reconstruction loss over the epoch.
    """
    model.train()
    total_loss     = 0.0
    n_batches      = 0
    grad_acc_steps = config["gradient_accumulation_steps"]
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, desc="MSM Pretrain", leave=False)):
        # wav from Dataset: (B, 3840, 2) → permute → (B, 2, 3840)
        wav = batch["wav"].to(device).permute(0, 2, 1)
        wav_masked, mask = apply_msm_mask(wav, config)

        with autocast(device_type=device.type):
            recon = model.forward_msm(wav_masked)
            loss  = msm_reconstruction_loss(recon, wav, mask) / grad_acc_steps

        amp_scaler.scale(loss).backward()
        if (step + 1) % grad_acc_steps == 0:
            amp_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["max_grad_norm"])
            amp_scaler.step(optimizer)
            amp_scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * grad_acc_steps
        n_batches  += 1

    return total_loss / max(n_batches, 1)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 14b. Supervised training epoch
# ─────────────────────────────────────────────────────────────────────────────
def train_one_epoch(
    model:            EEGModel,
    loader:           DataLoader,
    optimizer:        torch.optim.Optimizer,
    amp_scaler:       GradScaler,
    class_weights_nh: torch.Tensor,
    config:           Dict,
    device:           torch.device,
) -> float:
    """Run one supervised training epoch.

    Returns
    -------
    float
        Mean combined loss over the epoch.
    """
    model.train()
    total_loss     = 0.0
    n_batches      = 0
    grad_acc_steps = config["gradient_accumulation_steps"]
    class_weights_nh = class_weights_nh.to(device)
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, desc="Train", leave=False)):
        wav = batch["wav"].to(device).permute(0, 2, 1)   # (B, 2, 3840)
        x   = batch["x"].to(device)                      # (B, 1800, 14)
        x_spectral       = x[:, :, -CONFIG["transformer_n_spectral_features"]:]  # (B, 1800, 9)
        sqi_mask         = batch["sqi_flag"].to(device)                           # (B, 1800)
        modality_present = batch["modality_present"].squeeze(-1).to(device)       # (B,)
        label_ioh_5      = batch["label_ioh_5"].to(device)
        label_ioh_10     = batch["label_ioh_10"].to(device)
        label_ioh_15     = batch["label_ioh_15"].to(device)
        label_nh         = batch["label_nh"].squeeze(-1).to(device)
        sqi_end_gate     = sqi_mask[:, -1]                                        # (B,)

        with autocast(device_type=device.type):
            out = model(wav, x_spectral, sqi_mask, modality_present)
            loss_ioh5  = ioh_bce_loss(out["ioh_5"],  label_ioh_5,  sqi_end_gate)
            loss_ioh10 = ioh_bce_loss(out["ioh_10"], label_ioh_10, sqi_end_gate)
            loss_ioh15 = ioh_bce_loss(out["ioh_15"], label_ioh_15, sqi_end_gate)
            loss_nh    = focal_loss(
                out["nh"], label_nh,
                gamma=config["nh_focal_gamma"],
                class_weights=class_weights_nh,
                label_smoothing=config["nh_label_smoothing"],
            )
            loss = ((loss_ioh5 + loss_ioh10 + loss_ioh15) / 3.0 + loss_nh) / grad_acc_steps

        amp_scaler.scale(loss).backward()
        if (step + 1) % grad_acc_steps == 0:
            amp_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["max_grad_norm"])
            amp_scaler.step(optimizer)
            amp_scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * grad_acc_steps
        n_batches  += 1

    return total_loss / max(n_batches, 1)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 15. Validation
# ─────────────────────────────────────────────────────────────────────────────
def validate(
    model:            EEGModel,
    loader:           DataLoader,
    class_weights_nh: torch.Tensor,
    config:           Dict,
    device:           torch.device,
) -> Tuple[float, Dict[str, float]]:
    """Run one validation pass and return loss + metrics."""
    model.eval()
    total_loss       = 0.0
    n_batches        = 0
    class_weights_nh = class_weights_nh.to(device)

    all_ioh5_pred,  all_ioh10_pred,  all_ioh15_pred  = [], [], []
    all_ioh5_true,  all_ioh10_true,  all_ioh15_true  = [], [], []
    all_nh_pred,    all_nh_true                       = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Validate", leave=False):
            wav              = batch["wav"].to(device).permute(0, 2, 1)
            x                = batch["x"].to(device)
            x_spectral       = x[:, :, -CONFIG["transformer_n_spectral_features"]:]
            sqi_mask         = batch["sqi_flag"].to(device)
            modality_present = batch["modality_present"].squeeze(-1).to(device)
            label_ioh_5      = batch["label_ioh_5"].to(device)
            label_ioh_10     = batch["label_ioh_10"].to(device)
            label_ioh_15     = batch["label_ioh_15"].to(device)
            label_nh         = batch["label_nh"].squeeze(-1).to(device)
            sqi_end_gate     = sqi_mask[:, -1]

            with autocast(device_type=device.type):
                out        = model(wav, x_spectral, sqi_mask, modality_present)
                loss_ioh5  = ioh_bce_loss(out["ioh_5"],  label_ioh_5,  sqi_end_gate)
                loss_ioh10 = ioh_bce_loss(out["ioh_10"], label_ioh_10, sqi_end_gate)
                loss_ioh15 = ioh_bce_loss(out["ioh_15"], label_ioh_15, sqi_end_gate)
                loss_nh    = focal_loss(
                    out["nh"], label_nh,
                    gamma=config["nh_focal_gamma"],
                    class_weights=class_weights_nh,
                    label_smoothing=config["nh_label_smoothing"],
                )
                loss = (loss_ioh5 + loss_ioh10 + loss_ioh15) / 3.0 + loss_nh

            total_loss += loss.item()
            n_batches  += 1

            all_ioh5_pred.append(torch.sigmoid(out["ioh_5"]).cpu().numpy())
            all_ioh10_pred.append(torch.sigmoid(out["ioh_10"]).cpu().numpy())
            all_ioh15_pred.append(torch.sigmoid(out["ioh_15"]).cpu().numpy())
            all_ioh5_true.append(label_ioh_5.cpu().numpy())
            all_ioh10_true.append(label_ioh_10.cpu().numpy())
            all_ioh15_true.append(label_ioh_15.cpu().numpy())
            all_nh_pred.append(out["nh"].argmax(dim=-1).cpu().numpy())
            all_nh_true.append(label_nh.cpu().numpy())

    val_loss = total_loss / max(n_batches, 1)
    ioh_preds   = {k: np.concatenate(v).flatten() for k, v in
                   zip(["ioh_5","ioh_10","ioh_15"],[all_ioh5_pred,all_ioh10_pred,all_ioh15_pred])}
    ioh_targets = {k: np.concatenate(v).flatten() for k, v in
                   zip(["ioh_5","ioh_10","ioh_15"],[all_ioh5_true,all_ioh10_true,all_ioh15_true])}
    nh_preds    = np.concatenate(all_nh_pred)
    nh_targets  = np.concatenate(all_nh_true)

    metrics = compute_metrics(ioh_preds, ioh_targets, nh_preds, nh_targets)
    return val_loss, metrics

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 16. EarlyStopping
# ─────────────────────────────────────────────────────────────────────────────
class EarlyStopping:
    """Early stopping with best-model checkpointing.

    Stops training when val loss has not improved for `patience` epochs.
    """

    def __init__(self, patience: int = 8, checkpoint_path: str = "best_model.pt") -> None:
        self.patience        = patience
        self.checkpoint_path = checkpoint_path
        self.best_loss       = float("inf")
        self.counter         = 0
        self.should_stop     = False

    def step(self, val_loss: float, model: nn.Module) -> None:
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter   = 0
            torch.save(model.state_dict(), self.checkpoint_path)
            log.info(f"  ✓ New best val_loss={val_loss:.6f} — checkpoint saved")
        else:
            self.counter += 1
            log.info(f"  EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.should_stop = True
                log.info("  Early stopping triggered.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 17. train_model — orchestrates Stage 0 + Stage 1
# ─────────────────────────────────────────────────────────────────────────────
def train_model(
    model:            EEGModel,
    train_loader:     DataLoader,
    val_loader:       DataLoader,
    class_weights_nh: torch.Tensor,
    config:           Dict,
    device:           torch.device,
) -> EEGModel:
    """Train in two stages: MSM pretraining then supervised fine-tuning.

    Returns
    -------
    EEGModel
        Best model loaded from checkpoint.
    """
    model      = model.to(device)
    # Updated: GradScaler now requires device argument
    amp_scaler = GradScaler(device=device.type)

    # ── Stage 0: MSM pretraining ──────────────────────────────────────────────
    log.info("=== Stage 0: MSM Pretraining ===")
    optimizer_msm = torch.optim.Adam(
        model.parameters(), lr=config["msm_lr"], weight_decay=config["weight_decay"]
    )
    msm_loader = DataLoader(
        train_loader.dataset,
        batch_size=config["msm_batch_size"],
        shuffle=True, num_workers=0, pin_memory=True,
    )
    for epoch in range(1, config["msm_epochs"] + 1):
        msm_loss = pretrain_one_epoch_msm(
            model, msm_loader, optimizer_msm, amp_scaler, config, device
        )
        log.info(f"[MSM] Epoch {epoch}/{config['msm_epochs']}  loss={msm_loss:.6f}")

    torch.save(model.state_dict(), config["checkpoint_msm"])
    log.info(f"MSM checkpoint saved → {config['checkpoint_msm']}")

    # ── Stage 1: Supervised fine-tuning ──────────────────────────────────────
    log.info("=== Stage 1: Supervised Fine-tuning ===")
    optimizer_sup = torch.optim.AdamW(
        model.parameters(), lr=config["sup_lr"], weight_decay=config["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_sup, T_max=config["sup_epochs"], eta_min=1e-6
    )
    early_stop = EarlyStopping(
        patience=config["early_stopping_patience"],
        checkpoint_path=config["checkpoint_sup"],
    )

    for epoch in range(1, config["sup_epochs"] + 1):
        t0         = time.time()
        train_loss = train_one_epoch(
            model, train_loader, optimizer_sup, amp_scaler,
            class_weights_nh, config, device,
        )
        val_loss, val_metrics = validate(model, val_loader, class_weights_nh, config, device)
        scheduler.step()
        elapsed = time.time() - t0
        log.info(
            f"[Sup] Epoch {epoch}/{config['sup_epochs']}  "
            f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
            f"auroc_ioh5={val_metrics.get('auroc_ioh_5', float('nan')):.3f}  "
            f"nh_f1={val_metrics.get('nh_macro_f1', float('nan')):.3f}  "
            f"time={elapsed:.1f}s"
        )
        early_stop.step(val_loss, model)
        if early_stop.should_stop:
            break

    model.load_state_dict(torch.load(config["checkpoint_sup"], map_location=device,
                                     weights_only=True))
    log.info("Best supervised model loaded from checkpoint.")
    return model

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 18. Test evaluation
# ─────────────────────────────────────────────────────────────────────────────
def evaluate_test(
    model:            EEGModel,
    test_loader:      DataLoader,
    class_weights_nh: torch.Tensor,
    config:           Dict,
    device:           torch.device,
) -> Dict[str, float]:
    """Evaluate model on held-out test set and print full metrics."""
    test_loss, metrics = validate(model, test_loader, class_weights_nh, config, device)
    log.info("=== Test Set Evaluation ===")
    log.info(f"  Test loss: {test_loss:.6f}")
    for k, v in metrics.items():
        log.info(f"  {k}: {v:.4f}")
    return metrics

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 19. SHAP explanation
# ─────────────────────────────────────────────────────────────────────────────
def explain_with_shap(
    model:       EEGModel,
    test_loader: DataLoader,
    config:      Dict,
    device:      torch.device,
) -> None:
    """Generate SHAP feature importance plot for spectral + scalar features.

    Uses DeepExplainer on a spectral proxy forward pass → IOH-5 head.
    Full raw EEG SHAP is computationally infeasible; spectral features are used.
    Saves plot to config['shap_output'].
    """
    log.info("Computing SHAP values …")
    bg_x_list,   test_x_list = [], []
    max_bg, max_test          = 32, 64

    for batch in test_loader:
        x        = batch["x"].to(device)
        spectral = x[:, :, -CONFIG["transformer_n_spectral_features"]:]  # (B, 1800, 9)
        spectral_mean = spectral.mean(dim=1)                              # (B, 9)
        if len(bg_x_list) * spectral_mean.size(0) < max_bg:
            bg_x_list.append(spectral_mean.detach().cpu())
        if len(test_x_list) * spectral_mean.size(0) < max_test:
            test_x_list.append(spectral_mean.detach().cpu())

    if not bg_x_list or not test_x_list:
        log.warning("Not enough data for SHAP — skipping")
        return

    bg_x   = torch.cat(bg_x_list,   dim=0)[:max_bg]
    test_x = torch.cat(test_x_list, dim=0)[:max_test]

    class SpectralWrapper(nn.Module):
        def __init__(self, enc: EEGModel) -> None:
            super().__init__()
            self.enc = enc

        def forward(self, sp: torch.Tensor) -> torch.Tensor:
            # Expand (B, 9) → (B, N_vec, 9) and project through spectral_proj
            N_vec    = CONFIG["wavenet_n_output_vectors"]
            sp_exp   = sp.unsqueeze(1).expand(-1, N_vec, -1)
            proj     = self.enc.transformer_enc.spectral_proj(sp_exp)  # (B, N_vec, 128)
            emb      = proj.mean(dim=1)                                # (B, 128)
            return self.enc.ioh_head_5(emb)

    wrapper = SpectralWrapper(model).to(device).eval()

    try:
        explainer   = shap.DeepExplainer(wrapper, bg_x.to(device))
        shap_values = explainer.shap_values(test_x.to(device))

        feature_names = ["delta_power", "theta_power", "alpha_power", "beta_power",
                         "SEF95", "BSR", "BIS", "EMG", "SR"]
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.barh(feature_names, np.abs(np.array(shap_values)).mean(axis=0), color="steelblue")
        ax.set_xlabel("Mean |SHAP value|")
        ax.set_title("Model 3 — EEG Spectral Feature Importance (IOH-5 min)")
        plt.tight_layout()
        plt.savefig(config["shap_output"], dpi=150)
        plt.close()
        log.info(f"SHAP plot saved → {config['shap_output']}")
    except Exception as exc:
        log.warning(f"SHAP computation failed: {exc}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 20. Prediction visualisation
# ─────────────────────────────────────────────────────────────────────────────
def plot_predictions(
    model:       EEGModel,
    test_loader: DataLoader,
    config:      Dict,
    device:      torch.device,
    n_cases:     int = 5,
) -> None:
    """Plot IOH-5 predicted probability vs ground truth for n_cases samples.

    Saves to config['pred_output'].
    """
    model.eval()
    all_pred, all_true, all_caseids = [], [], []

    with torch.no_grad():
        for batch in test_loader:
            wav              = batch["wav"].to(device).permute(0, 2, 1)
            x                = batch["x"].to(device)
            x_spectral       = x[:, :, -CONFIG["transformer_n_spectral_features"]:]
            sqi_mask         = batch["sqi_flag"].to(device)
            modality_present = batch["modality_present"].squeeze(-1).to(device)

            out        = model(wav, x_spectral, sqi_mask, modality_present)
            pred_proba = torch.sigmoid(out["ioh_5"]).cpu().numpy().flatten()
            true_label = batch["label_ioh_5"].cpu().numpy().flatten()
            cids       = batch["caseid"]

            all_pred.extend(pred_proba.tolist())
            all_true.extend(true_label.tolist())
            all_caseids.extend(cids.tolist() if isinstance(cids, torch.Tensor) else list(cids))

            if len(all_pred) >= n_cases * 50:
                break

    unique_cases = list(dict.fromkeys(all_caseids))[:n_cases]
    fig, axes   = plt.subplots(n_cases, 1, figsize=(12, 3 * n_cases), squeeze=False)

    for ax_row, cid in zip(axes, unique_cases):
        ax     = ax_row[0]
        idxs   = [i for i, c in enumerate(all_caseids) if c == cid]
        pred_c = [all_pred[i] for i in idxs]
        true_c = [all_true[i] for i in idxs]
        t      = list(range(len(pred_c)))
        ax.plot(t, pred_c, label="P(IOH-5min)", color="steelblue")
        ax.fill_between(t, true_c, alpha=0.3, color="red", label="IOH true")
        ax.set_ylim(0, 1)
        ax.set_ylabel("Prob")
        ax.set_title(f"Case {cid}")
        ax.legend(loc="upper right", fontsize=8)

    axes[-1][0].set_xlabel("Window index")
    plt.suptitle("Model 3 — EEG IOH-5 Predictions vs Ground Truth", fontsize=13)
    plt.tight_layout()
    plt.savefig(config["pred_output"], dpi=150)
    plt.close()
    log.info(f"Prediction plot saved → {config['pred_output']}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 21. Main entry point
# ─────────────────────────────────────────────────────────────────────────────
def main() -> None:
    """End-to-end pipeline: data loading, preprocessing, training, evaluation."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log.info(f"Using device: {device}")

    # ── 4a: Clinical metadata ─────────────────────────────────────────────────
    clinical_df = load_clinical_metadata(CONFIG)

    # ── Determine valid case IDs (BIS monitor cases only) ────────────────────
    try:
        bis_cases = set(vitaldb.caseids_bis)
        log.info(f"vitaldb.caseids_bis: {len(bis_cases)} cases with BIS monitor")
    except AttributeError:
        log.warning("vitaldb.caseids_bis not available — iterating all 6388 cases")
        bis_cases = set(range(1, 6389))

    valid_caseids = sorted(bis_cases)
    log.info(f"Attempting to load {len(valid_caseids)} BIS cases")

    # ── 4b: Load waveform data ─────────────────────────────────────────────────
    wav_data, scalar_data = load_waveform_data(valid_caseids, CONFIG)

    # ── 8: Split cases BEFORE window generation (no leakage) ──────────────────
    all_loaded_caseids = sorted(set(wav_data.keys()) & set(scalar_data.keys()))
    train_caseids, val_caseids, test_caseids = split_cases(
        all_loaded_caseids,
        val_frac=CONFIG["val_frac"],
        test_frac=CONFIG["test_frac"],
        seed=CONFIG["seed"],
    )

    # ── 6: Window generation per split ────────────────────────────────────────
    def _subset(src_dict, keys):
        return {c: src_dict[c] for c in keys if c in src_dict}

    log.info("Generating training windows …")
    train_windows = generate_windows(
        _subset(wav_data, train_caseids), _subset(scalar_data, train_caseids),
        clinical_df, CONFIG
    )
    gc.collect()

    log.info("Generating validation windows …")
    val_windows = generate_windows(
        _subset(wav_data, val_caseids), _subset(scalar_data, val_caseids),
        clinical_df, CONFIG
    )
    gc.collect()

    log.info("Generating test windows …")
    test_windows = generate_windows(
        _subset(wav_data, test_caseids), _subset(scalar_data, test_caseids),
        clinical_df, CONFIG
    )
    del wav_data, scalar_data
    gc.collect()

    if not train_windows:
        log.error("No training windows generated — aborting")
        return

    # ── 7: Fit scalers on training data only ─────────────────────────────────
    scaler_scalar, scaler_spectral = fit_scalers(train_windows)
    train_windows = apply_scalers(train_windows, scaler_scalar, scaler_spectral)
    val_windows   = apply_scalers(val_windows,   scaler_scalar, scaler_spectral)
    test_windows  = apply_scalers(test_windows,  scaler_scalar, scaler_spectral)

    # ── Class weights on training N/H labels ──────────────────────────────────
    class_weights_nh = compute_class_weights(
        [w["label_nh"] for w in train_windows], n_classes=CONFIG["nh_n_classes"]
    )
    log.info(f"N/H class weights: {class_weights_nh.tolist()}")

    # ── 9–10: Datasets and DataLoaders ───────────────────────────────────────
    train_ds = EEGDataset(train_windows)
    val_ds   = EEGDataset(val_windows)
    test_ds  = EEGDataset(test_windows)
    train_loader, val_loader, test_loader = build_dataloaders(train_ds, val_ds, test_ds, CONFIG)

    # ── 11: Model instantiation ───────────────────────────────────────────────
    model    = EEGModel(CONFIG)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log.info(f"EEGModel parameters: {n_params:,}")

    # ── 17: Train (Stage 0 MSM + Stage 1 supervised) ─────────────────────────
    model = train_model(model, train_loader, val_loader, class_weights_nh, CONFIG, device)

    # ── 18: Test evaluation ───────────────────────────────────────────────────
    evaluate_test(model, test_loader, class_weights_nh, CONFIG, device)

    # ── 19: SHAP explanation ──────────────────────────────────────────────────
    explain_with_shap(model, test_loader, CONFIG, device)

    # ── 20: Prediction visualisation ─────────────────────────────────────────
    plot_predictions(model, test_loader, CONFIG, device, n_cases=5)

    log.info("model_3 pipeline complete.")


if __name__ == "__main__":
    main()

In [ ]:
# Run the full pipeline
main()